# 06 — Feature Engineering

## Purpose

This notebook transforms the validated WAHIS–climate integrated dataset into
a modelling-ready feature table for poultry disease early-warning analysis.

The feature-engineering process will create:

1. Temporal features
2. Geographic features
3. Disease-related features
4. Climate features
5. Historical outbreak features
6. Data-availability and missingness indicators

## Methodological principles

- Missing or unavailable outbreak observations are **not treated as zero**.
- Historical outbreak features must use only information available before the
  prediction period.
- Future observations must never be used to construct predictors.
- Climate variables are retained as continuous measurements.
- Climate-unavailable observations remain distinguishable from measured values.
- Disease-specific historical features will be constructed within the relevant
  ADM1 and disease groups.
- Feature engineering will preserve the original analytical record identity.
- All transformations must be reproducible and validated before modelling.

## Input

`data/processed/wahis_climate_integrated.csv`

## Expected output

`data/processed/wahis_climate_features.csv`

In [1]:
# ============================================================
# PROJECT SETUP AND DATA LOADING
# Notebook: 06_Feature_Engineering.ipynb
# ============================================================

from pathlib import Path

import numpy as np
import pandas as pd


# ------------------------------------------------------------
# 1. Define project directories
# ------------------------------------------------------------
# The notebook is stored inside the "notebooks" directory.
# Therefore, the project root is the parent directory of the
# current working directory.

PROJECT_ROOT = Path.cwd().parent

DATA_DIR = PROJECT_ROOT / "data"
PROCESSED_DIR = DATA_DIR / "processed"

INPUT_FILE = PROCESSED_DIR / "wahis_climate_integrated.csv"
OUTPUT_FILE = PROCESSED_DIR / "wahis_climate_features.csv"


# ------------------------------------------------------------
# 2. Validate the input file
# ------------------------------------------------------------
if not INPUT_FILE.exists():
    raise FileNotFoundError(
        f"Integrated dataset was not found:\n{INPUT_FILE}"
    )


# ------------------------------------------------------------
# 3. Load the validated integrated dataset
# ------------------------------------------------------------
df = pd.read_csv(INPUT_FILE)


# ------------------------------------------------------------
# 4. Display basic structure
# ------------------------------------------------------------
print("=" * 70)
print("FEATURE ENGINEERING — DATA LOADING")
print("=" * 70)

print("\nProject root:")
print(PROJECT_ROOT)

print("\nInput file:")
print(INPUT_FILE)

print("\nDataset shape:")
print(f"Rows: {df.shape[0]:,}")
print(f"Columns: {df.shape[1]:,}")


# ------------------------------------------------------------
# 5. Display column groups
# ------------------------------------------------------------
print("\nColumn names:")
print(df.columns.tolist())


# ------------------------------------------------------------
# 6. Inspect the first records
# ------------------------------------------------------------
print("\nFirst five records:")
display(df.head())


# ------------------------------------------------------------
# 7. Confirm critical columns exist
# ------------------------------------------------------------
required_columns = [
    "analytical_record_id",
    "year",
    "semester",
    "administrative_division",
    "disease",
    "new_outbreaks",
    "outbreak_count_observed",
    "outbreak_observation_status",
    "adm1_key",
    "year_key",
    "semester_key",
    "integration_key",
    "temperature_c",
    "dewpoint_c",
    "semester_total_precipitation_mm",
    "semester_mean_wind_component_magnitude_ms",
    "climate_match_status"
]

missing_required_columns = [
    column
    for column in required_columns
    if column not in df.columns
]

print("\nRequired-column validation:")
if missing_required_columns:
    print("Missing columns:")
    for column in missing_required_columns:
        print(f"  - {column}")

    raise ValueError(
        "The integrated dataset is missing required columns."
    )

print("All required columns are present.")


# ------------------------------------------------------------
# 8. Confirm analytical record identity
# ------------------------------------------------------------
record_id_missing = df["analytical_record_id"].isna().sum()
record_id_duplicates = df["analytical_record_id"].duplicated().sum()

print("\nAnalytical record validation:")
print(f"Missing record IDs: {record_id_missing:,}")
print(f"Duplicate record IDs: {record_id_duplicates:,}")


# ------------------------------------------------------------
# 9. Confirm outbreak observation logic
# ------------------------------------------------------------
print("\nOutbreak observation status:")
print(
    df["outbreak_observation_status"]
    .value_counts(dropna=False)
    .to_string()
)


# ------------------------------------------------------------
# 10. Confirm climate coverage
# ------------------------------------------------------------
print("\nClimate match status:")
print(
    df["climate_match_status"]
    .value_counts(dropna=False)
    .to_string()
)


# ------------------------------------------------------------
# 11. Final loading validation
# ------------------------------------------------------------
loading_checks = {
    "Input dataset contains 1,180 records":
        len(df) == 1180,

    "Analytical record IDs are unique":
        record_id_duplicates == 0,

    "No analytical record IDs are missing":
        record_id_missing == 0,

    "All required columns are present":
        len(missing_required_columns) == 0,
}

print("\n" + "=" * 70)
print("DATA LOADING VALIDATION")
print("=" * 70)

for check, passed in loading_checks.items():
    status = "PASSED" if passed else "FAILED"
    print(f"{status}: {check}")

if not all(loading_checks.values()):
    raise ValueError(
        "Feature-engineering data loading validation failed."
    )

print("\nFeature-engineering dataset loaded successfully.")

FEATURE ENGINEERING — DATA LOADING

Project root:
c:\Users\TECHGROW ANALYTICS\poultry-disease-early-warning-nigeria

Input file:
c:\Users\TECHGROW ANALYTICS\poultry-disease-early-warning-nigeria\data\processed\wahis_climate_integrated.csv

Dataset shape:
Rows: 1,180
Columns: 39

Column names:
['analytical_record_id', 'year', 'semester', 'country', 'world_region', 'administrative_division', 'disease', 'animal_category', 'species', 'new_outbreaks', 'outbreak_count_observed', 'outbreak_observation_status', 'serotype_subtype_genotype', 'event_id', 'outbreak_id', 'susceptible', 'measuring_units', 'cases', 'killed_and_disposed_of', 'slaughtered', 'deaths', 'vaccinated', 'validated_boundary_name', 'geographic_level', 'geographic_match_status', 'adm1_key', 'year_key', 'semester_key', 'integration_key', 'months_observed', 'temperature_c', 'dewpoint_c', 'semester_total_precipitation_mm', 'semester_mean_wind_component_magnitude_ms', 'minimum_temperature_c_spatial_coverage', 'minimum_dewpoint_c_sp

,analytical_record_id,year,semester,country,world_region,administrative_division,disease,animal_category,species,new_outbreaks,...,months_observed,temperature_c,dewpoint_c,semester_total_precipitation_mm,semester_mean_wind_component_magnitude_ms,minimum_temperature_c_spatial_coverage,minimum_dewpoint_c_spatial_coverage,minimum_precipitation_monthly_total_mm_spatial_coverage,minimum_wind_speed_from_mean_components_ms_spatial_coverage,climate_match_status
0,1,2005,Jul-Dec 2005,Nigeria,Africa,Nigeria,Fowl pox (-2005),Domestic,Birds,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Unavailable
1,2,2005,Jul-Dec 2005,Nigeria,Africa,Nigeria,Fowl typhoid,Domestic,Birds,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Unavailable
2,3,2005,Jul-Dec 2005,Nigeria,Africa,Nigeria,Infectious bursal disease (Gumboro disease),Domestic,Birds,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Unavailable
3,4,2005,Jul-Dec 2005,Nigeria,Africa,Nigeria,Newcastle disease virus (Inf. with),Domestic,Birds,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Unavailable
4,5,2005,Jan-Jun 2005,Nigeria,Africa,Nigeria,Newcastle disease virus (Inf. with),Domestic,Birds,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Unavailable



Required-column validation:
All required columns are present.

Analytical record validation:
Missing record IDs: 0
Duplicate record IDs: 0

Outbreak observation status:
outbreak_observation_status
Unavailable    735
Observed       445

Climate match status:
climate_match_status
Matched        984
Unavailable    196

DATA LOADING VALIDATION
PASSED: Input dataset contains 1,180 records
PASSED: Analytical record IDs are unique
PASSED: No analytical record IDs are missing
PASSED: All required columns are present

Feature-engineering dataset loaded successfully.


In [3]:
# ==============================================================
# CELL 3 — VALIDATE THE MODELLING GRAIN
# ==============================================================

# This cell establishes the temporal and geographic grain that will
# be used for feature engineering.
#
# Important:
# - WAHIS contains both ADM1-level and country-level records.
# - Country-level records must NOT be deleted simply because they
#   cannot receive ADM1-level climate features.
# - Historical disease features will later be calculated using
#   ADM1 + disease + chronological period where ADM1 is available.
# - Missing ADM1 information will remain explicitly identifiable.

print("=" * 70)
print("FEATURE ENGINEERING — MODELLING GRAIN VALIDATION")
print("=" * 70)

# --------------------------------------------------------------
# 1. Create a standardised semester variable
# --------------------------------------------------------------

# The original semester column contains values such as:
# "Jan-Jun 2019" and "Jul-Dec 2020".
#
# We extract only the semester component because year is already
# stored separately in the dataset.

df["semester_model"] = (
    df["semester"]
    .astype("string")
    .str.extract(r"^(Jan-Jun|Jul-Dec)", expand=False)
)

print("\nSemester modelling values:")
print(df["semester_model"].value_counts(dropna=False).to_string())

# --------------------------------------------------------------
# 2. Create chronological ordering variables
# --------------------------------------------------------------

# Assign a numeric order to each semester:
#
# Jan-Jun = 1
# Jul-Dec = 2

semester_order = {
    "Jan-Jun": 1,
    "Jul-Dec": 2
}

df["semester_order"] = df["semester_model"].map(semester_order)

# Create a continuous half-year period.
#
# This gives us a reliable chronological variable for later
# lag and rolling-window calculations.

df["temporal_period"] = (
    df["year"] * 2 + (df["semester_order"] - 1)
)

print("\nTemporal ordering:")
print(
    f"Minimum temporal period: {df['temporal_period'].min()}"
)
print(
    f"Maximum temporal period: {df['temporal_period'].max()}"
)

# --------------------------------------------------------------
# 3. Check missing values in the modelling dimensions
# --------------------------------------------------------------

grain_columns = [
    "adm1_key",
    "disease",
    "year",
    "semester_model"
]

print("\nMissing values in modelling dimensions:")
print(
    df[grain_columns]
    .isna()
    .sum()
    .to_string()
)

# --------------------------------------------------------------
# 4. Examine geographic levels
# --------------------------------------------------------------

print("\nGeographic level distribution:")
print(
    df["geographic_level"]
    .value_counts(dropna=False)
    .to_string()
)

print("\nGeographic match status:")
print(
    df["geographic_match_status"]
    .value_counts(dropna=False)
    .to_string()
)

# --------------------------------------------------------------
# 5. Identify country-level records
# --------------------------------------------------------------

country_level_mask = (
    df["geographic_level"].eq("Country")
)

country_level_records = df.loc[
    country_level_mask,
    [
        "analytical_record_id",
        "administrative_division",
        "validated_boundary_name",
        "geographic_level",
        "adm1_key",
        "climate_match_status"
    ]
]

print("\nCountry-level records:")
print(
    f"Number of country-level records: "
    f"{country_level_mask.sum():,}"
)

if country_level_mask.sum() > 0:
    display(country_level_records.head(20))

# --------------------------------------------------------------
# 6. Check the ADM1 modelling grain
# --------------------------------------------------------------

# Disease-specific historical features will use:
#
#     ADM1 + Disease + Year + Semester
#
# We therefore examine whether this combination is unique.

adm1_grain_columns = [
    "adm1_key",
    "disease",
    "year",
    "semester_model"
]

adm1_grain = (
    df.loc[
        df["adm1_key"].notna(),
        adm1_grain_columns
    ]
    .drop_duplicates()
)

print("\nADM1 modelling grain:")
print(
    f"Unique ADM1 × disease × year × semester combinations: "
    f"{len(adm1_grain):,}"
)

# Count records belonging to each ADM1/disease/time combination.

grain_counts = (
    df.loc[df["adm1_key"].notna()]
    .groupby(adm1_grain_columns, dropna=False)
    .size()
    .reset_index(name="record_count")
)

repeated_grain = (
    grain_counts[
        grain_counts["record_count"] > 1
    ]
    .sort_values(
        ["record_count", "year", "adm1_key", "disease"],
        ascending=[False, True, True, True]
    )
)

print("\nRepeated ADM1 modelling combinations:")
print(
    f"Number of repeated combinations: "
    f"{len(repeated_grain):,}"
)

if len(repeated_grain) > 0:
    print("\nTop repeated ADM1 modelling combinations:")
    display(repeated_grain.head(20))

# --------------------------------------------------------------
# 7. Inspect one repeated combination
# --------------------------------------------------------------

# Repeated combinations are not automatically errors.
# They may occur because WAHIS contains multiple records for the
# same disease, geography and semester, potentially representing
# different events or reporting records.

if len(repeated_grain) > 0:

    example = repeated_grain.iloc[0]

    example_mask = (
        df["adm1_key"].eq(example["adm1_key"]) &
        df["disease"].eq(example["disease"]) &
        df["year"].eq(example["year"]) &
        df["semester_model"].eq(example["semester_model"])
    )

    example_records = df.loc[
        example_mask,
        [
            "analytical_record_id",
            "adm1_key",
            "disease",
            "year",
            "semester_model",
            "new_outbreaks",
            "outbreak_count_observed",
            "outbreak_observation_status",
            "event_id",
            "outbreak_id",
            "animal_category",
            "species"
        ]
    ].sort_values("analytical_record_id")

    print("\nExample repeated ADM1 modelling combination:")
    print({
        "adm1_key": example["adm1_key"],
        "disease": example["disease"],
        "year": example["year"],
        "semester": example["semester_model"]
    })

    print("\nRecords belonging to this combination:")
    display(example_records)

# --------------------------------------------------------------
# 8. Validate temporal structure
# --------------------------------------------------------------

temporal_checks = {
    "Semester extraction successful": (
        df["semester_model"].notna().all()
    ),
    "Semester order successfully assigned": (
        df["semester_order"].notna().all()
    ),
    "Temporal period successfully created": (
        df["temporal_period"].notna().all()
    )
}

print("\nTemporal validation:")
for check, passed in temporal_checks.items():
    print(
        f"{'PASSED' if passed else 'FAILED'}: {check}"
    )

# --------------------------------------------------------------
# 9. Validate disease and year dimensions
# --------------------------------------------------------------

disease_missing = df["disease"].isna().sum()
year_missing = df["year"].isna().sum()

print("\nCore dimension validation:")
print(f"Missing disease values: {disease_missing:,}")
print(f"Missing year values: {year_missing:,}")

# --------------------------------------------------------------
# 10. Final modelling-grain assessment
# --------------------------------------------------------------

# ADM1 is intentionally allowed to be missing for country-level
# records. This is a valid geographic condition rather than a
# data-quality failure.

final_checks = {
    "Semester values successfully extracted": (
        df["semester_model"].notna().all()
    ),
    "Semester ordering successfully created": (
        df["semester_order"].notna().all()
    ),
    "Temporal periods successfully created": (
        df["temporal_period"].notna().all()
    ),
    "Disease values are available": (
        disease_missing == 0
    ),
    "Year values are available": (
        year_missing == 0
    ),
    "Analytical record IDs remain unique": (
        df["analytical_record_id"].duplicated().sum() == 0
    ),
    "Country-level records are explicitly identifiable": (
        country_level_mask.sum() >= 0
    )
}

print("\n" + "=" * 70)
print("MODELLING-GRAIN VALIDATION")
print("=" * 70)

for check, passed in final_checks.items():
    status = "PASSED" if passed else "FAILED"
    print(f"{status}: {check}")

if not all(final_checks.values()):
    raise ValueError(
        "Modelling-grain validation failed. "
        "Review the failed checks before proceeding."
    )

print("\n" + "=" * 70)
print("MODELLING-GRAIN VALIDATION COMPLETED")
print("=" * 70)

print(
    "\nImportant methodological decision:"
)
print(
    "Country-level records are retained in the dataset, "
    "but ADM1-specific features will only be calculated "
    "where a validated ADM1 is available."
)

print(
    "\nThe dataset is ready for the next feature-engineering stage."
)

FEATURE ENGINEERING — MODELLING GRAIN VALIDATION

Semester modelling values:
semester_model
Jul-Dec    629
Jan-Jun    551

Temporal ordering:
Minimum temporal period: 4010
Maximum temporal period: 4051

Missing values in modelling dimensions:
adm1_key          1
disease           0
year              0
semester_model    0

Geographic level distribution:
geographic_level
ADM1       1174
Country       5
NaN           1

Geographic match status:
geographic_match_status
Matched                                            1174
Country-level — excluded from ADM1 spatial join       5
NaN                                                   1

Country-level records:
Number of country-level records: 5


,analytical_record_id,administrative_division,validated_boundary_name,geographic_level,adm1_key,climate_match_status
0,1,Nigeria,Nigeria,Country,Nigeria,Unavailable
1,2,Nigeria,Nigeria,Country,Nigeria,Unavailable
2,3,Nigeria,Nigeria,Country,Nigeria,Unavailable
3,4,Nigeria,Nigeria,Country,Nigeria,Unavailable
4,5,Nigeria,Nigeria,Country,Nigeria,Unavailable



ADM1 modelling grain:
Unique ADM1 × disease × year × semester combinations: 738

Repeated ADM1 modelling combinations:
Number of repeated combinations: 439

Top repeated ADM1 modelling combinations:


,adm1_key,disease,year,semester_model,record_count
346,Kaduna,Infectious bursal disease (Gumboro disease),2020,Jul-Dec,3
315,Jigawa,Infectious bursal disease (Gumboro disease),2021,Jan-Jun,3
9,Abia,Newcastle disease virus (Inf. with),2020,Jul-Dec,2
65,Anambra,Newcastle disease virus (Inf. with),2020,Jul-Dec,2
104,Bayelsa,Newcastle disease virus (Inf. with),2020,Jan-Jun,2
105,Bayelsa,Newcastle disease virus (Inf. with),2020,Jul-Dec,2
146,Borno,Infectious bursal disease (Gumboro disease),2020,Jul-Dec,2
148,Borno,Newcastle disease virus (Inf. with),2020,Jan-Jun,2
149,Borno,Newcastle disease virus (Inf. with),2020,Jul-Dec,2
157,Cross River,Fowl typhoid,2020,Jan-Jun,2



Example repeated ADM1 modelling combination:
{'adm1_key': 'Kaduna', 'disease': 'Infectious bursal disease (Gumboro disease)', 'year': np.int64(2020), 'semester': 'Jul-Dec'}

Records belonging to this combination:


,analytical_record_id,adm1_key,disease,year,semester_model,new_outbreaks,outbreak_count_observed,outbreak_observation_status,event_id,outbreak_id,animal_category,species
257,258,Kaduna,Infectious bursal disease (Gumboro disease),2020,Jul-Dec,NaN,False,Unavailable,NaN,NaN,Domestic,Birds
258,259,Kaduna,Infectious bursal disease (Gumboro disease),2020,Jul-Dec,2.0,True,Observed,NaN,NaN,Domestic,NaN
259,260,Kaduna,Infectious bursal disease (Gumboro disease),2020,Jul-Dec,NaN,False,Unavailable,NaN,NaN,Domestic,Birds



Temporal validation:
PASSED: Semester extraction successful
PASSED: Semester order successfully assigned
PASSED: Temporal period successfully created

Core dimension validation:
Missing disease values: 0
Missing year values: 0

MODELLING-GRAIN VALIDATION
PASSED: Semester values successfully extracted
PASSED: Semester ordering successfully created
PASSED: Temporal periods successfully created
PASSED: Disease values are available
PASSED: Year values are available
PASSED: Analytical record IDs remain unique
PASSED: Country-level records are explicitly identifiable

MODELLING-GRAIN VALIDATION COMPLETED

Important methodological decision:
Country-level records are retained in the dataset, but ADM1-specific features will only be calculated where a validated ADM1 is available.

The dataset is ready for the next feature-engineering stage.


In [4]:
# ==============================================================
# CELL 4 — CREATE TEMPORAL FEATURES
# ==============================================================

# This cell creates modelling-ready temporal features.
#
# Important methodological principles:
#
# 1. Temporal features must be derived only from the observation's
#    own date/period.
# 2. No outbreak information is used to create these features.
# 3. Semester is represented both as a categorical/indicator variable
#    and as cyclical features.
# 4. The temporal_period variable will later be used to ensure that
#    historical outbreak features are strictly chronological.

print("=" * 70)
print("FEATURE ENGINEERING — TEMPORAL FEATURES")
print("=" * 70)

# --------------------------------------------------------------
# 1. Validate the temporal variables created in Cell 3
# --------------------------------------------------------------

required_temporal_columns = [
    "year",
    "semester_model",
    "semester_order",
    "temporal_period"
]

missing_temporal_columns = [
    column
    for column in required_temporal_columns
    if column not in df.columns
]

if missing_temporal_columns:
    raise ValueError(
        "The following temporal columns are missing: "
        + ", ".join(missing_temporal_columns)
    )

# --------------------------------------------------------------
# 2. Create semester indicator variables
# --------------------------------------------------------------

# Binary indicators make the semester information directly usable
# by many machine-learning models.

df["is_first_semester"] = (
    df["semester_model"].eq("Jan-Jun").astype(int)
)

df["is_second_semester"] = (
    df["semester_model"].eq("Jul-Dec").astype(int)
)

# --------------------------------------------------------------
# 3. Create cyclical semester features
# --------------------------------------------------------------

# A simple numeric encoding such as:
#
# Jan-Jun = 1
# Jul-Dec = 2
#
# implies that the distance between semesters is meaningful.
# Cyclical encoding instead represents the semester as a position
# within a repeating cycle.

df["semester_sin"] = np.sin(
    2 * np.pi * (df["semester_order"] - 1) / 2
)

df["semester_cos"] = np.cos(
    2 * np.pi * (df["semester_order"] - 1) / 2
)

# --------------------------------------------------------------
# 4. Create long-term year trend features
# --------------------------------------------------------------

# The dataset spans multiple years. We retain the original year
# and create a simple relative time variable beginning at 2019,
# which is the start of the climate-data period.

df["years_since_2019"] = df["year"] - 2019

# --------------------------------------------------------------
# 5. Create cyclical annual features
# --------------------------------------------------------------

# These features represent the position of each observation within
# the annual time cycle.
#
# Because the observations are semester-based, the year itself is
# treated as a long-term trend while the semester variables capture
# the within-year cycle.

# We use the year trend as the primary long-term temporal component.
# The cyclical annual features below are retained as optional
# modelling features for models that can benefit from smooth
# temporal representations.

year_min = df["year"].min()
year_max = df["year"].max()

year_range = year_max - year_min

if year_range > 0:

    year_position = (
        (df["year"] - year_min) / year_range
    )

    df["year_sin"] = np.sin(
        2 * np.pi * year_position
    )

    df["year_cos"] = np.cos(
        2 * np.pi * year_position
    )

else:

    # This branch protects the code if the dataset unexpectedly
    # contains only one year.

    df["year_sin"] = 0.0
    df["year_cos"] = 1.0

# --------------------------------------------------------------
# 6. Validate temporal feature values
# --------------------------------------------------------------

temporal_feature_columns = [
    "is_first_semester",
    "is_second_semester",
    "semester_sin",
    "semester_cos",
    "years_since_2019",
    "year_sin",
    "year_cos"
]

print("\nCreated temporal features:")
for column in temporal_feature_columns:
    print(f"  - {column}")

# Check for missing values in the newly created features.

temporal_missing = (
    df[temporal_feature_columns]
    .isna()
    .sum()
)

print("\nMissing values in temporal features:")
print(temporal_missing.to_string())

# Validate binary semester indicators.

semester_indicator_check = (
    df["is_first_semester"].isin([0, 1]).all()
    and df["is_second_semester"].isin([0, 1]).all()
)

# Validate that exactly one semester indicator is active
# for each record.

semester_exclusivity_check = (
    (
        df["is_first_semester"]
        + df["is_second_semester"]
    ).eq(1).all()
)

# Validate temporal ordering.

temporal_order_check = (
    df["temporal_period"].notna().all()
)

# --------------------------------------------------------------
# 7. Display a sample of the engineered features
# --------------------------------------------------------------

sample_columns = [
    "analytical_record_id",
    "year",
    "semester",
    "semester_model",
    "semester_order",
    "temporal_period",
    "is_first_semester",
    "is_second_semester",
    "semester_sin",
    "semester_cos",
    "years_since_2019",
    "year_sin",
    "year_cos"
]

print("\nSample of temporal features:")
display(
    df[sample_columns]
    .sort_values(
        ["year", "semester_order", "analytical_record_id"]
    )
    .head(12)
)

# --------------------------------------------------------------
# 8. Final temporal-feature validation
# --------------------------------------------------------------

temporal_checks = {
    "No missing temporal feature values": (
        temporal_missing.sum() == 0
    ),
    "Semester indicators contain only 0/1": (
        semester_indicator_check
    ),
    "Exactly one semester indicator is active per record": (
        semester_exclusivity_check
    ),
    "Temporal ordering is available for every record": (
        temporal_order_check
    ),
    "Analytical record IDs remain unique": (
        df["analytical_record_id"].duplicated().sum() == 0
    )
}

print("\n" + "=" * 70)
print("TEMPORAL FEATURE VALIDATION")
print("=" * 70)

for check, passed in temporal_checks.items():
    status = "PASSED" if passed else "FAILED"
    print(f"{status}: {check}")

if not all(temporal_checks.values()):
    raise ValueError(
        "Temporal feature validation failed. "
        "Review the temporal feature construction before proceeding."
    )

print("\nTemporal feature engineering completed successfully.")

FEATURE ENGINEERING — TEMPORAL FEATURES

Created temporal features:
  - is_first_semester
  - is_second_semester
  - semester_sin
  - semester_cos
  - years_since_2019
  - year_sin
  - year_cos

Missing values in temporal features:
is_first_semester     0
is_second_semester    0
semester_sin          0
semester_cos          0
years_since_2019      0
year_sin              0
year_cos              0

Sample of temporal features:


,analytical_record_id,year,semester,semester_model,semester_order,temporal_period,is_first_semester,is_second_semester,semester_sin,semester_cos,years_since_2019,year_sin,year_cos
4,5,2005,Jan-Jun 2005,Jan-Jun,1,4010,1,0,0.000000e+00,1.0,-14,0.000000,1.000000
0,1,2005,Jul-Dec 2005,Jul-Dec,2,4011,0,1,1.224647e-16,-1.0,-14,0.000000,1.000000
1,2,2005,Jul-Dec 2005,Jul-Dec,2,4011,0,1,1.224647e-16,-1.0,-14,0.000000,1.000000
2,3,2005,Jul-Dec 2005,Jul-Dec,2,4011,0,1,1.224647e-16,-1.0,-14,0.000000,1.000000
3,4,2005,Jul-Dec 2005,Jul-Dec,2,4011,0,1,1.224647e-16,-1.0,-14,0.000000,1.000000
5,6,2007,Jul-Dec 2007,Jul-Dec,2,4015,0,1,1.224647e-16,-1.0,-12,0.587785,0.809017
6,7,2008,Jan-Jun 2008,Jan-Jun,1,4016,1,0,0.000000e+00,1.0,-11,0.809017,0.587785
7,8,2008,Jan-Jun 2008,Jan-Jun,1,4016,1,0,0.000000e+00,1.0,-11,0.809017,0.587785
8,9,2008,Jan-Jun 2008,Jan-Jun,1,4016,1,0,0.000000e+00,1.0,-11,0.809017,0.587785
9,10,2008,Jan-Jun 2008,Jan-Jun,1,4016,1,0,0.000000e+00,1.0,-11,0.809017,0.587785



TEMPORAL FEATURE VALIDATION
PASSED: No missing temporal feature values
PASSED: Semester indicators contain only 0/1
PASSED: Exactly one semester indicator is active per record
PASSED: Temporal ordering is available for every record
PASSED: Analytical record IDs remain unique

Temporal feature engineering completed successfully.


In [5]:
# ==============================================================
# CELL 5 — CREATE GEOGRAPHIC AND DISEASE FEATURES
# ==============================================================

# This cell creates basic geographic and disease descriptors.
#
# These features describe the observation itself and do not use
# future outbreak information.
#
# Important:
# - Country-level records are retained.
# - Missing ADM1 is represented explicitly.
# - Disease categories are encoded only as descriptive identifiers
#   at this stage.
# - Target-derived historical outbreak features are NOT created here.

print("=" * 70)
print("FEATURE ENGINEERING — GEOGRAPHIC AND DISEASE FEATURES")
print("=" * 70)

# --------------------------------------------------------------
# 1. Validate required source columns
# --------------------------------------------------------------

required_columns = [
    "adm1_key",
    "disease",
    "geographic_level",
    "analytical_record_id"
]

missing_columns = [
    column
    for column in required_columns
    if column not in df.columns
]

if missing_columns:
    raise ValueError(
        "Missing required columns: "
        + ", ".join(missing_columns)
    )

# --------------------------------------------------------------
# 2. Create ADM1 availability indicator
# --------------------------------------------------------------

# This explicitly identifies whether an observation has a validated
# ADM1 geography available for spatial modelling.

df["has_adm1"] = (
    df["adm1_key"].notna().astype(int)
)

# --------------------------------------------------------------
# 3. Create country-level indicator
# --------------------------------------------------------------

# Country-level records are retained rather than discarded.
# This indicator makes their geographic level explicit.

df["is_country_level"] = (
    df["geographic_level"].eq("Country").astype(int)
)

# --------------------------------------------------------------
# 4. Create a disease code
# --------------------------------------------------------------

# Convert the disease category into a stable integer identifier.
#
# The mapping is created from the sorted unique disease names so
# that the same disease receives the same code throughout this
# feature-engineering run.
#
# This code is a categorical identifier, not a numerical measure
# of disease severity or importance.

disease_categories = sorted(
    df["disease"].dropna().unique()
)

disease_code_mapping = {
    disease: code
    for code, disease in enumerate(
        disease_categories,
        start=1
    )
}

df["disease_code"] = (
    df["disease"]
    .map(disease_code_mapping)
    .astype("Int64")
)

# --------------------------------------------------------------
# 5. Calculate disease frequency
# --------------------------------------------------------------

# Disease frequency describes how many analytical records belong
# to each disease in the current dataset.
#
# This is a descriptive feature and does not use outbreak counts.

disease_frequency_mapping = (
    df["disease"]
    .value_counts()
    .to_dict()
)

df["disease_frequency"] = (
    df["disease"]
    .map(disease_frequency_mapping)
)

# --------------------------------------------------------------
# 6. Calculate ADM1 × disease record count
# --------------------------------------------------------------

# This feature describes how many WAHIS analytical records exist
# for a given ADM1 and disease combination.
#
# It does NOT use outbreak counts.

df["adm1_disease_record_count"] = (
    df.groupby(
        ["adm1_key", "disease"],
        dropna=False
    )["analytical_record_id"]
    .transform("count")
)

# --------------------------------------------------------------
# 7. Display geographic feature summary
# --------------------------------------------------------------

print("\nGeographic feature summary:")

print(
    f"Records with validated ADM1: "
    f"{df['has_adm1'].sum():,}"
)

print(
    f"Records without ADM1: "
    f"{(df['has_adm1'] == 0).sum():,}"
)

print(
    f"Country-level records: "
    f"{df['is_country_level'].sum():,}"
)

print("\nGeographic level distribution:")
print(
    df["geographic_level"]
    .value_counts(dropna=False)
    .to_string()
)

# --------------------------------------------------------------
# 8. Display disease feature summary
# --------------------------------------------------------------

print("\nNumber of unique diseases:")
print(
    df["disease"].nunique()
)

print("\nDisease categories and codes:")

disease_summary = (
    df[
        ["disease", "disease_code"]
    ]
    .drop_duplicates()
    .sort_values("disease_code")
)

display(disease_summary)

# --------------------------------------------------------------
# 9. Display disease frequency
# --------------------------------------------------------------

print("\nDisease record frequencies:")

disease_frequency_summary = (
    df.groupby(
        ["disease", "disease_code"],
        dropna=False
    )
    .size()
    .reset_index(name="record_count")
    .sort_values(
        "record_count",
        ascending=False
    )
)

display(disease_frequency_summary)

# --------------------------------------------------------------
# 10. Validate newly created features
# --------------------------------------------------------------

new_feature_columns = [
    "has_adm1",
    "is_country_level",
    "disease_code",
    "disease_frequency",
    "adm1_disease_record_count"
]

print("\nMissing values in newly created features:")
print(
    df[new_feature_columns]
    .isna()
    .sum()
    .to_string()
)

# ADM1 availability should be binary.

adm1_indicator_check = (
    df["has_adm1"].isin([0, 1]).all()
)

# Country-level indicator should also be binary.

country_indicator_check = (
    df["is_country_level"].isin([0, 1]).all()
)

# Every disease should have a valid disease code.

disease_code_check = (
    df["disease_code"].notna().all()
)

# Disease frequency should be positive for every record.

disease_frequency_check = (
    df["disease_frequency"].notna().all()
    and (df["disease_frequency"] > 0).all()
)

# ADM1 × disease record count should be positive for every record.

adm1_disease_count_check = (
    df["adm1_disease_record_count"].notna().all()
    and (df["adm1_disease_record_count"] > 0).all()
)

# Analytical record identity must remain unchanged.

record_id_check = (
    df["analytical_record_id"].duplicated().sum() == 0
)

# --------------------------------------------------------------
# 11. Final validation
# --------------------------------------------------------------

geographic_disease_checks = {
    "ADM1 indicator contains only 0/1": (
        adm1_indicator_check
    ),
    "Country-level indicator contains only 0/1": (
        country_indicator_check
    ),
    "Every disease has a disease code": (
        disease_code_check
    ),
    "Disease frequency is available and positive": (
        disease_frequency_check
    ),
    "ADM1 × disease record count is valid": (
        adm1_disease_count_check
    ),
    "Analytical record IDs remain unique": (
        record_id_check
    )
}

print("\n" + "=" * 70)
print("GEOGRAPHIC AND DISEASE FEATURE VALIDATION")
print("=" * 70)

for check, passed in geographic_disease_checks.items():
    status = "PASSED" if passed else "FAILED"
    print(f"{status}: {check}")

if not all(geographic_disease_checks.values()):
    raise ValueError(
        "Geographic and disease feature validation failed. "
        "Review the failed checks before proceeding."
    )

print("\nGeographic and disease feature engineering completed successfully.")

FEATURE ENGINEERING — GEOGRAPHIC AND DISEASE FEATURES

Geographic feature summary:
Records with validated ADM1: 1,179
Records without ADM1: 1
Country-level records: 5

Geographic level distribution:
geographic_level
ADM1       1174
Country       5
NaN           1

Number of unique diseases:
6

Disease categories and codes:


,disease,disease_code
251,Avian infectious bronchitis,1
42,Fowl cholera (-2011),2
0,Fowl pox (-2005),3
1,Fowl typhoid,4
2,Infectious bursal disease (Gumboro disease),5
3,Newcastle disease virus (Inf. with),6



Disease record frequencies:


,disease,disease_code,record_count
5,Newcastle disease virus (Inf. with),6,610
4,Infectious bursal disease (Gumboro disease),5,324
3,Fowl typhoid,4,192
0,Avian infectious bronchitis,1,51
1,Fowl cholera (-2011),2,2
2,Fowl pox (-2005),3,1



Missing values in newly created features:
has_adm1                     0
is_country_level             0
disease_code                 0
disease_frequency            0
adm1_disease_record_count    0

GEOGRAPHIC AND DISEASE FEATURE VALIDATION
PASSED: ADM1 indicator contains only 0/1
PASSED: Country-level indicator contains only 0/1
PASSED: Every disease has a disease code
PASSED: Disease frequency is available and positive
PASSED: ADM1 × disease record count is valid
PASSED: Analytical record IDs remain unique

Geographic and disease feature engineering completed successfully.


In [9]:
# ------------------------------------------------------------
# 7. Display sample of engineered climate features
# ------------------------------------------------------------

# Define the columns that should be displayed.
# semester_order is intentionally NOT included in the final display
# because it is only being used to establish chronological ordering.

climate_display_columns = [
    "analytical_record_id",
    "year",
    "semester",
    "administrative_division",
    "disease",
    "climate_match_status",
    "temperature_c",
    "dewpoint_c",
    "semester_total_precipitation_mm",
    "semester_mean_wind_component_magnitude_ms",
    "climate_variable_count",
    "climate_complete"
]

# Sort the original dataframe first, while semester_order still exists,
# and then select only the columns intended for display.
climate_display = (
    df.sort_values(
        ["year", "semester_order", "analytical_record_id"]
    )[climate_display_columns]
    .head(15)
)

display(climate_display)


print("\n" + "=" * 70)
print("CLIMATE FEATURE ENGINEERING VALIDATION PASSED")
print("=" * 70)

,analytical_record_id,year,semester,administrative_division,disease,climate_match_status,temperature_c,dewpoint_c,semester_total_precipitation_mm,semester_mean_wind_component_magnitude_ms,climate_variable_count,climate_complete
0,1,2005,Jul-Dec 2005,Nigeria,Fowl pox (-2005),Unavailable,NaN,NaN,NaN,NaN,0,0
1,2,2005,Jul-Dec 2005,Nigeria,Fowl typhoid,Unavailable,NaN,NaN,NaN,NaN,0,0
2,3,2005,Jul-Dec 2005,Nigeria,Infectious bursal disease (Gumboro disease),Unavailable,NaN,NaN,NaN,NaN,0,0
3,4,2005,Jul-Dec 2005,Nigeria,Newcastle disease virus (Inf. with),Unavailable,NaN,NaN,NaN,NaN,0,0
4,5,2005,Jan-Jun 2005,Nigeria,Newcastle disease virus (Inf. with),Unavailable,NaN,NaN,NaN,NaN,0,0
5,6,2007,Jul-Dec 2007,Gombe,Infectious bursal disease (Gumboro disease),Unavailable,NaN,NaN,NaN,NaN,0,0
6,7,2008,Jan-Jun 2008,Adamawa,Newcastle disease virus (Inf. with),Unavailable,NaN,NaN,NaN,NaN,0,0
7,8,2008,Jan-Jun 2008,Bauchi,Infectious bursal disease (Gumboro disease),Unavailable,NaN,NaN,NaN,NaN,0,0
8,9,2008,Jan-Jun 2008,Bauchi,Newcastle disease virus (Inf. with),Unavailable,NaN,NaN,NaN,NaN,0,0
9,10,2008,Jan-Jun 2008,Delta,Fowl typhoid,Unavailable,NaN,NaN,NaN,NaN,0,0



CLIMATE FEATURE ENGINEERING VALIDATION PASSED


In [13]:
# ============================================================
# DIAGNOSTIC CELL — OUTBREAK OBSERVATION STATUS
# Notebook: 06_Feature_Engineering.ipynb
# ============================================================

# This diagnostic checks the actual relationship between:
#   - outbreak_observation_status
#   - outbreak_count_observed
#   - new_outbreaks
#
# We will use the result to construct the historical features
# correctly without making assumptions about unavailable values.

print("=" * 70)
print("OUTBREAK OBSERVATION STATUS DIAGNOSTIC")
print("=" * 70)

print("\n1. Outbreak observation status:")
print(
    df["outbreak_observation_status"]
    .value_counts(dropna=False)
    .to_string()
)

print("\n2. outbreak_count_observed data type:")
print(df["outbreak_count_observed"].dtype)

print("\n3. outbreak_count_observed missingness:")
print(
    df["outbreak_count_observed"]
    .isna()
    .value_counts()
    .rename(
        index={
            False: "Not missing",
            True: "Missing"
        }
    )
    .to_string()
)

print("\n4. Cross-tabulation:")
print(
    pd.crosstab(
        df["outbreak_observation_status"],
        df["outbreak_count_observed"].isna(),
        margins=True
    )
)

print("\n5. Cross-tabulation using new_outbreaks:")
print(
    pd.crosstab(
        df["outbreak_observation_status"],
        df["new_outbreaks"].isna(),
        margins=True
    )
)

print("\n6. Sample records marked 'Unavailable':")

display(
    df.loc[
        df["outbreak_observation_status"] == "Unavailable",
        [
            "analytical_record_id",
            "year",
            "semester",
            "administrative_division",
            "disease",
            "new_outbreaks",
            "outbreak_count_observed",
            "outbreak_observation_status"
        ]
    ].head(15)
)

print("\n7. Sample records marked 'Observed':")

display(
    df.loc[
        df["outbreak_observation_status"] == "Observed",
        [
            "analytical_record_id",
            "year",
            "semester",
            "administrative_division",
            "disease",
            "new_outbreaks",
            "outbreak_count_observed",
            "outbreak_observation_status"
        ]
    ].head(15)
)

print("\n" + "=" * 70)
print("DIAGNOSTIC COMPLETE")
print("=" * 70)

OUTBREAK OBSERVATION STATUS DIAGNOSTIC

1. Outbreak observation status:
outbreak_observation_status
Unavailable    735
Observed       445

2. outbreak_count_observed data type:
bool

3. outbreak_count_observed missingness:
outbreak_count_observed
Not missing    1180

4. Cross-tabulation:
outbreak_count_observed      False   All
outbreak_observation_status             
Observed                       445   445
Unavailable                    735   735
All                           1180  1180

5. Cross-tabulation using new_outbreaks:
new_outbreaks                False  True   All
outbreak_observation_status                   
Observed                       445     0   445
Unavailable                      0   735   735
All                            445   735  1180

6. Sample records marked 'Unavailable':


,analytical_record_id,year,semester,administrative_division,disease,new_outbreaks,outbreak_count_observed,outbreak_observation_status
0,1,2005,Jul-Dec 2005,Nigeria,Fowl pox (-2005),NaN,False,Unavailable
1,2,2005,Jul-Dec 2005,Nigeria,Fowl typhoid,NaN,False,Unavailable
2,3,2005,Jul-Dec 2005,Nigeria,Infectious bursal disease (Gumboro disease),NaN,False,Unavailable
3,4,2005,Jul-Dec 2005,Nigeria,Newcastle disease virus (Inf. with),NaN,False,Unavailable
4,5,2005,Jan-Jun 2005,Nigeria,Newcastle disease virus (Inf. with),NaN,False,Unavailable
5,6,2007,Jul-Dec 2007,Gombe,Infectious bursal disease (Gumboro disease),NaN,False,Unavailable
6,7,2008,Jan-Jun 2008,Adamawa,Newcastle disease virus (Inf. with),NaN,False,Unavailable
7,8,2008,Jan-Jun 2008,Bauchi,Infectious bursal disease (Gumboro disease),NaN,False,Unavailable
8,9,2008,Jan-Jun 2008,Bauchi,Newcastle disease virus (Inf. with),NaN,False,Unavailable
9,10,2008,Jan-Jun 2008,Delta,Fowl typhoid,NaN,False,Unavailable



7. Sample records marked 'Observed':


,analytical_record_id,year,semester,administrative_division,disease,new_outbreaks,outbreak_count_observed,outbreak_observation_status
210,211,2020,Jan-Jun 2020,Bayelsa,Newcastle disease virus (Inf. with),6.0,True,Observed
213,214,2020,Jan-Jun 2020,Borno,Newcastle disease virus (Inf. with),3.0,True,Observed
215,216,2020,Jan-Jun 2020,Cross River,Fowl typhoid,1.0,True,Observed
217,218,2020,Jan-Jun 2020,Jigawa,Newcastle disease virus (Inf. with),5.0,True,Observed
220,221,2020,Jan-Jun 2020,Kaduna,Newcastle disease virus (Inf. with),3.0,True,Observed
222,223,2020,Jan-Jun 2020,Katsina,Newcastle disease virus (Inf. with),1.0,True,Observed
224,225,2020,Jan-Jun 2020,Niger,Fowl typhoid,1.0,True,Observed
227,228,2020,Jan-Jun 2020,Niger,Newcastle disease virus (Inf. with),5.0,True,Observed
229,230,2020,Jan-Jun 2020,Ondo,Newcastle disease virus (Inf. with),2.0,True,Observed
232,233,2020,Jan-Jun 2020,Osun,Newcastle disease virus (Inf. with),3.0,True,Observed



DIAGNOSTIC COMPLETE


In [17]:
# ============================================================
# CELL 7 — SEMESTER-LEVEL OUTBREAK HISTORY FEATURES
# ============================================================
#
# Purpose:
# Create leakage-safe historical outbreak features at the
# ADM1 + disease + semester level.
#
# IMPORTANT:
# Multiple WAHIS records can exist within the same
# ADM1 + disease + year + semester.
#
# Therefore, historical features are first constructed on a
# unique semester-level table and then merged back to the
# original WAHIS records.
#
# Missing/unavailable outbreak observations are NOT converted
# to zero.
# ============================================================

import numpy as np
import pandas as pd


# ------------------------------------------------------------
# 1. Validate required columns
# ------------------------------------------------------------

required_columns = [
    "analytical_record_id",
    "year",
    "semester",
    "semester_period",
    "semester_order",
    "temporal_order",
    "adm1_key",
    "disease",
    "new_outbreaks",
    "outbreak_observation_status",
]

missing_columns = [
    col for col in required_columns
    if col not in df.columns
]

assert not missing_columns, (
    f"Missing required columns: {missing_columns}"
)


# ------------------------------------------------------------
# 2. Create safe outbreak-count field
# ------------------------------------------------------------

# Only observed outbreak counts are used.
# Unavailable observations remain NaN.

df["outbreak_count_for_history"] = df["new_outbreaks"].where(
    df["outbreak_observation_status"] == "Observed"
)

assert (
    df.loc[
        df["outbreak_observation_status"] == "Observed",
        "outbreak_count_for_history"
    ].notna().all()
), "Observed records contain missing outbreak counts."

assert (
    df.loc[
        df["outbreak_observation_status"] == "Unavailable",
        "outbreak_count_for_history"
    ].isna().all()
), "Unavailable records must remain missing."


# ------------------------------------------------------------
# 3. Define semester-level modelling key
# ------------------------------------------------------------

period_columns = [
    "adm1_key",
    "disease",
    "year",
    "semester_period",
    "semester_order",
    "temporal_order",
]


# ------------------------------------------------------------
# 4. Create one record per ADM1 + disease + semester
# ------------------------------------------------------------

semester_outbreaks = (
    df.groupby(
        period_columns,
        dropna=False,
        as_index=False
    )
    .agg(
        semester_outbreak_count=(
            "outbreak_count_for_history",
            lambda x: x.sum(min_count=1)
        ),
        observed_outbreak_records=(
            "outbreak_count_for_history",
            lambda x: x.notna().sum()
        ),
        unavailable_outbreak_records=(
            "outbreak_count_for_history",
            lambda x: x.isna().sum()
        ),
        total_wahis_records=(
            "analytical_record_id",
            "count"
        ),
    )
)


# ------------------------------------------------------------
# 5. Identify whether the semester has an observed count
# ------------------------------------------------------------

semester_outbreaks["semester_outbreak_observed"] = (
    semester_outbreaks[
        "semester_outbreak_count"
    ].notna().astype(int)
)


# ------------------------------------------------------------
# 6. Sort chronologically
# ------------------------------------------------------------

semester_outbreaks = semester_outbreaks.sort_values(
    by=[
        "adm1_key",
        "disease",
        "temporal_order",
    ],
    na_position="last"
).reset_index(drop=True)


# ------------------------------------------------------------
# 7. Create previous-semester temporal reference
# ------------------------------------------------------------

history_group = [
    "adm1_key",
    "disease",
]

# Store the temporal order of the source period.
#
# This allows us to validate that historical features really
# come from an earlier period.

semester_outbreaks[
    "previous_semester_temporal_order"
] = (
    semester_outbreaks
    .groupby(history_group, dropna=False)
    ["temporal_order"]
    .shift(1)
)


# ------------------------------------------------------------
# 8. Create previous-semester outbreak count
# ------------------------------------------------------------

semester_outbreaks[
    "previous_semester_outbreak_count"
] = (
    semester_outbreaks
    .groupby(history_group, dropna=False)
    ["semester_outbreak_count"]
    .shift(1)
)

semester_outbreaks[
    "previous_semester_outbreak_available"
] = (
    semester_outbreaks[
        "previous_semester_outbreak_count"
    ].notna().astype(int)
)


# ------------------------------------------------------------
# 9. Create previous-semester status
# ------------------------------------------------------------

semester_outbreaks[
    "previous_semester_outbreak_status"
] = np.where(
    semester_outbreaks[
        "previous_semester_outbreak_count"
    ].notna(),
    "Observed",
    "Unavailable"
)


# ------------------------------------------------------------
# 10. Create two-semesters-prior feature
# ------------------------------------------------------------

semester_outbreaks[
    "two_semesters_prior_outbreak_count"
] = (
    semester_outbreaks
    .groupby(history_group, dropna=False)
    ["semester_outbreak_count"]
    .shift(2)
)

semester_outbreaks[
    "two_semesters_prior_available"
] = (
    semester_outbreaks[
        "two_semesters_prior_outbreak_count"
    ].notna().astype(int)
)


# ------------------------------------------------------------
# 11. Create previous-three-semester rolling features
# ------------------------------------------------------------

previous_semester_counts = (
    semester_outbreaks
    .groupby(history_group, dropna=False)
    ["semester_outbreak_count"]
    .shift(1)
)

semester_outbreaks[
    "outbreak_history_mean_3"
] = (
    previous_semester_counts
    .groupby(
        [
            semester_outbreaks["adm1_key"],
            semester_outbreaks["disease"],
        ],
        dropna=False
    )
    .transform(
        lambda x: x.rolling(
            window=3,
            min_periods=1
        ).mean()
    )
)

semester_outbreaks[
    "outbreak_history_max_3"
] = (
    previous_semester_counts
    .groupby(
        [
            semester_outbreaks["adm1_key"],
            semester_outbreaks["disease"],
        ],
        dropna=False
    )
    .transform(
        lambda x: x.rolling(
            window=3,
            min_periods=1
        ).max()
    )
)


# ------------------------------------------------------------
# 12. Create availability indicators
# ------------------------------------------------------------

semester_outbreaks[
    "outbreak_history_mean_3_available"
] = (
    semester_outbreaks[
        "outbreak_history_mean_3"
    ].notna().astype(int)
)

semester_outbreaks[
    "outbreak_history_max_3_available"
] = (
    semester_outbreaks[
        "outbreak_history_max_3"
    ].notna().astype(int)
)


# ------------------------------------------------------------
# 13. Validate the semester-level history table
# ------------------------------------------------------------

history_key = [
    "adm1_key",
    "disease",
    "year",
    "semester_period",
]

duplicate_history_keys = (
    semester_outbreaks
    .duplicated(subset=history_key)
    .sum()
)

assert duplicate_history_keys == 0, (
    "Duplicate ADM1 + disease + semester periods detected."
)


# ------------------------------------------------------------
# 14. Validate temporal ordering
# ------------------------------------------------------------

# This is the correct leakage test.
#
# Whenever a previous-semester reference exists, its temporal
# order must be strictly less than the current period.

valid_previous_reference = (
    semester_outbreaks[
        "previous_semester_temporal_order"
    ].notna()
)

temporal_leakage_count = (
    (
        semester_outbreaks.loc[
            valid_previous_reference,
            "previous_semester_temporal_order"
        ]
        >=
        semester_outbreaks.loc[
            valid_previous_reference,
            "temporal_order"
        ]
    )
).sum()

assert temporal_leakage_count == 0, (
    "Temporal leakage detected: historical period is not "
    "strictly earlier than the current period."
)


# ------------------------------------------------------------
# 15. Prepare historical features for merging
# ------------------------------------------------------------

history_features = semester_outbreaks[
    [
        "adm1_key",
        "disease",
        "year",
        "semester_period",

        "semester_outbreak_count",
        "observed_outbreak_records",
        "unavailable_outbreak_records",
        "total_wahis_records",
        "semester_outbreak_observed",

        "previous_semester_outbreak_count",
        "previous_semester_outbreak_available",
        "previous_semester_outbreak_status",

        "two_semesters_prior_outbreak_count",
        "two_semesters_prior_available",

        "outbreak_history_mean_3",
        "outbreak_history_max_3",
        "outbreak_history_mean_3_available",
        "outbreak_history_max_3_available",

        # Keep this for final leakage validation.
        "previous_semester_temporal_order",
    ]
].copy()


# ------------------------------------------------------------
# 16. Remove previous row-level history features
# ------------------------------------------------------------

old_history_columns = [
    "outbreak_lag_1",
    "outbreak_lag_1_available",
    "outbreak_lag_2",
    "outbreak_lag_2_available",
    "historical_outbreak_observed",
    "historical_outbreak_positive",
    "historical_outbreak_zero",
    "outbreak_history_mean_3",
    "outbreak_history_max_3",
    "outbreak_history_mean_3_available",
    "outbreak_history_max_3_available",
]

df = df.drop(
    columns=[
        col
        for col in old_history_columns
        if col in df.columns
    ]
)


# ------------------------------------------------------------
# 17. Merge semester-level history back to WAHIS records
# ------------------------------------------------------------

merge_keys = [
    "adm1_key",
    "disease",
    "year",
    "semester_period",
]

records_before_merge = len(df)

df = df.merge(
    history_features,
    on=merge_keys,
    how="left",
    validate="many_to_one",
    suffixes=("", "_history")
)

assert len(df) == records_before_merge, (
    "History merge changed the number of WAHIS records."
)


# ------------------------------------------------------------
# 18. Validate historical temporal ordering after merge
# ------------------------------------------------------------

merged_temporal_check = (
    df["previous_semester_temporal_order"].notna()
)

post_merge_leakage_count = (
    (
        df.loc[
            merged_temporal_check,
            "previous_semester_temporal_order"
        ]
        >=
        df.loc[
            merged_temporal_check,
            "temporal_order"
        ]
    )
).sum()

assert post_merge_leakage_count == 0, (
    "Temporal leakage detected after merging historical features."
)


# ------------------------------------------------------------
# 19. Validate unavailable periods
# ------------------------------------------------------------

fully_unavailable_periods = semester_outbreaks[
    semester_outbreaks[
        "observed_outbreak_records"
    ] == 0
]

assert (
    fully_unavailable_periods[
        "semester_outbreak_count"
    ].isna().all()
), (
    "Periods with no observed outbreak counts "
    "must remain missing."
)


# ------------------------------------------------------------
# 20. Calculate summary statistics
# ------------------------------------------------------------

total_periods = len(semester_outbreaks)

observed_periods = int(
    semester_outbreaks[
        "semester_outbreak_observed"
    ].sum()
)

unavailable_periods = (
    total_periods - observed_periods
)

previous_available = int(
    df["previous_semester_outbreak_available"].sum()
)

two_prior_available = int(
    df["two_semesters_prior_available"].sum()
)

rolling_mean_available = int(
    df["outbreak_history_mean_3_available"].sum()
)

rolling_max_available = int(
    df["outbreak_history_max_3_available"].sum()
)


# ------------------------------------------------------------
# 21. Display validation results
# ------------------------------------------------------------

print("=" * 70)
print("SEMESTER-LEVEL OUTBREAK HISTORY FEATURE ENGINEERING")
print("=" * 70)

print(
    f"\nOriginal WAHIS records: "
    f"{records_before_merge:,}"
)

print(
    f"Final WAHIS records: "
    f"{len(df):,}"
)

print(
    f"\nUnique ADM1 + disease + semester periods: "
    f"{total_periods:,}"
)

print(
    f"Periods with observed outbreak counts: "
    f"{observed_periods:,}"
)

print(
    f"Periods without observed outbreak counts: "
    f"{unavailable_periods:,}"
)

print("\nHistorical feature availability:")

print(
    f"  Previous semester: "
    f"{previous_available:,} available / "
    f"{len(df):,} total"
)

print(
    f"  Two semesters prior: "
    f"{two_prior_available:,} available / "
    f"{len(df):,} total"
)

print(
    f"  Previous 3-semester mean: "
    f"{rolling_mean_available:,} available / "
    f"{len(df):,} total"
)

print(
    f"  Previous 3-semester maximum: "
    f"{rolling_max_available:,} available / "
    f"{len(df):,} total"
)

print(
    f"\nTemporal leakage records: "
    f"{post_merge_leakage_count}"
)


# ------------------------------------------------------------
# 22. Display representative records
# ------------------------------------------------------------

display_columns = [
    "analytical_record_id",
    "year",
    "semester",
    "adm1_key",
    "disease",
    "new_outbreaks",
    "outbreak_observation_status",

    "semester_outbreak_count",
    "previous_semester_outbreak_count",
    "previous_semester_outbreak_available",

    "two_semesters_prior_outbreak_count",
    "outbreak_history_mean_3",
    "outbreak_history_max_3",
]

print("\nSample engineered records:")

display(
    df.sort_values(
        [
            "adm1_key",
            "disease",
            "temporal_order",
            "analytical_record_id",
        ],
        na_position="last"
    )[display_columns].head(25)
)


print("\n" + "=" * 70)
print("SEMESTER-LEVEL OUTBREAK HISTORY VALIDATION PASSED")
print("=" * 70)

SEMESTER-LEVEL OUTBREAK HISTORY FEATURE ENGINEERING

Original WAHIS records: 1,180
Final WAHIS records: 1,180

Unique ADM1 + disease + semester periods: 739
Periods with observed outbreak counts: 445
Periods without observed outbreak counts: 294

Historical feature availability:
  Previous semester: 663 available / 1,180 total
  Two semesters prior: 534 available / 1,180 total
  Previous 3-semester mean: 741 available / 1,180 total
  Previous 3-semester maximum: 741 available / 1,180 total

Temporal leakage records: 0

Sample engineered records:


,analytical_record_id,year,semester,adm1_key,disease,new_outbreaks,outbreak_observation_status,semester_outbreak_count,previous_semester_outbreak_count,previous_semester_outbreak_available,two_semesters_prior_outbreak_count,outbreak_history_mean_3,outbreak_history_max_3
0,715,2023,Jan-Jun 2023,Abia,Avian infectious bronchitis,NaN,Unavailable,NaN,NaN,0,NaN,NaN,NaN
1,375,2021,Jul-Dec 2021,Abia,Fowl typhoid,7.0,Observed,7.0,NaN,0,NaN,NaN,NaN
2,376,2021,Jul-Dec 2021,Abia,Fowl typhoid,NaN,Unavailable,7.0,NaN,0,NaN,NaN,NaN
3,992,2025,Jan-Jun 2025,Abia,Fowl typhoid,1.0,Observed,1.0,7.0,1,NaN,7.000000,7.0
4,993,2025,Jan-Jun 2025,Abia,Fowl typhoid,NaN,Unavailable,1.0,7.0,1,NaN,7.000000,7.0
5,285,2021,Jan-Jun 2021,Abia,Infectious bursal disease (Gumboro disease),2.0,Observed,2.0,NaN,0,NaN,NaN,NaN
6,286,2021,Jan-Jun 2021,Abia,Infectious bursal disease (Gumboro disease),NaN,Unavailable,2.0,NaN,0,NaN,NaN,NaN
7,377,2021,Jul-Dec 2021,Abia,Infectious bursal disease (Gumboro disease),5.0,Observed,5.0,2.0,1,NaN,2.000000,2.0
8,378,2021,Jul-Dec 2021,Abia,Infectious bursal disease (Gumboro disease),NaN,Unavailable,5.0,2.0,1,NaN,2.000000,2.0
9,509,2022,Jan-Jun 2022,Abia,Infectious bursal disease (Gumboro disease),2.0,Observed,2.0,5.0,1,2.0,3.500000,5.0



SEMESTER-LEVEL OUTBREAK HISTORY VALIDATION PASSED


In [18]:
# ======================================================================
# CELL 8 — TARGET DEFINITION AND MODELLING ELIGIBILITY
# ======================================================================
# Purpose:
#   Define the modelling target without treating unavailable outbreak
#   observations as zero.
#
# Important methodological rule:
#   - Observed outbreak counts are valid target observations.
#   - Unavailable outbreak counts remain unavailable.
#   - We do NOT convert unavailable values to zero.
#   - Historical features remain predictors only when they are based on
#     information from earlier periods.
# ======================================================================

# ----------------------------------------------------------------------
# 1. Define the modelling target
# ----------------------------------------------------------------------

# The observed outbreak count is the supervised target only when WAHIS
# provides an actual numeric outbreak count.
df["target_outbreak_count"] = df["new_outbreaks"].where(
    df["outbreak_observation_status"] == "Observed"
)

# Create a simple modelling eligibility indicator.
# A record is eligible for supervised outbreak-count modelling only
# when the target is actually observed.
df["target_observed"] = df["target_outbreak_count"].notna()


# ----------------------------------------------------------------------
# 2. Create a binary outbreak target
# ----------------------------------------------------------------------
# Because all currently observed outbreak counts are positive, a binary
# outbreak-occurrence target can be derived safely for observed records.
#
# Unavailable observations remain NaN rather than becoming 0.

df["target_outbreak_occurrence"] = np.where(
    df["target_observed"],
    (df["target_outbreak_count"] > 0).astype(int),
    np.nan
)


# ----------------------------------------------------------------------
# 3. Create target availability/status fields
# ----------------------------------------------------------------------

df["target_status"] = np.where(
    df["target_observed"],
    "Observed",
    "Unavailable"
)


# ----------------------------------------------------------------------
# 4. Validate the target construction
# ----------------------------------------------------------------------

assert df["target_outbreak_count"].notna().sum() == (
    df["outbreak_observation_status"] == "Observed"
).sum(), (
    "Target count does not match observed outbreak status."
)

assert df.loc[
    df["target_observed"],
    "target_outbreak_count"
].notna().all(), (
    "Observed target records contain missing outbreak counts."
)

assert df.loc[
    ~df["target_observed"],
    "target_outbreak_count"
].isna().all(), (
    "Unavailable outbreak observations were incorrectly assigned a target."
)

# Confirm that unavailable observations have not been converted into zero.
unavailable_target_zero_count = (
    (~df["target_observed"])
    & (df["target_outbreak_count"] == 0)
).sum()

assert unavailable_target_zero_count == 0, (
    "Unavailable outbreak observations were incorrectly converted to zero."
)


# ----------------------------------------------------------------------
# 5. Validate binary target
# ----------------------------------------------------------------------

observed_binary_values = set(
    df.loc[
        df["target_observed"],
        "target_outbreak_occurrence"
    ].dropna().unique()
)

assert observed_binary_values.issubset({0, 1}), (
    "Binary outbreak target contains unexpected values."
)

unavailable_binary_count = df[
    ~df["target_observed"]
]["target_outbreak_occurrence"].notna().sum()

assert unavailable_binary_count == 0, (
    "Unavailable observations received a binary target."
)


# ----------------------------------------------------------------------
# 6. Summarise modelling eligibility
# ----------------------------------------------------------------------

target_summary = pd.DataFrame({
    "metric": [
        "Total WAHIS records",
        "Observed outbreak targets",
        "Unavailable outbreak targets",
        "Observed target percentage",
        "Positive outbreak targets",
        "Zero outbreak targets",
    ],
    "value": [
        len(df),
        df["target_observed"].sum(),
        (~df["target_observed"]).sum(),
        round(df["target_observed"].mean() * 100, 2),
        (
            df.loc[
                df["target_observed"],
                "target_outbreak_occurrence"
            ] == 1
        ).sum(),
        (
            df.loc[
                df["target_observed"],
                "target_outbreak_occurrence"
            ] == 0
        ).sum(),
    ]
})

print("=" * 70)
print("TARGET DEFINITION AND MODELLING ELIGIBILITY")
print("=" * 70)

print("\nTarget count status:")
print(df["target_status"].value_counts())

print("\nTarget outbreak-count summary:")
print(
    df.loc[
        df["target_observed"],
        "target_outbreak_count"
    ].describe()
)

print("\nBinary outbreak target:")
print(
    df.loc[
        df["target_observed"],
        "target_outbreak_occurrence"
    ].value_counts()
    .sort_index()
)

print("\nModelling eligibility summary:")
display(target_summary)

print("\nSample target records:")
display(
    df[
        [
            "analytical_record_id",
            "year",
            "semester",
            "adm1_key",
            "disease",
            "new_outbreaks",
            "outbreak_observation_status",
            "target_outbreak_count",
            "target_outbreak_occurrence",
            "target_status",
        ]
    ]
    .sort_values(
        ["year", "semester", "adm1_key", "disease"]
    )
    .head(25)
)

print("\n" + "=" * 70)
print("TARGET DEFINITION VALIDATION PASSED")
print("=" * 70)

TARGET DEFINITION AND MODELLING ELIGIBILITY

Target count status:
target_status
Unavailable    735
Observed       445
Name: count, dtype: int64

Target outbreak-count summary:
count    445.000000
mean       6.840449
std       12.872353
min        1.000000
25%        1.000000
50%        3.000000
75%        7.000000
max      184.000000
Name: target_outbreak_count, dtype: float64

Binary outbreak target:
target_outbreak_occurrence
1.0    445
Name: count, dtype: int64

Modelling eligibility summary:


,metric,value
0,Total WAHIS records,1180.00
1,Observed outbreak targets,445.00
2,Unavailable outbreak targets,735.00
3,Observed target percentage,37.71
4,Positive outbreak targets,445.00
5,Zero outbreak targets,0.00



Sample target records:


,analytical_record_id,year,semester,adm1_key,disease,new_outbreaks,outbreak_observation_status,target_outbreak_count,target_outbreak_occurrence,target_status
854,5,2005,Jan-Jun 2005,Nigeria,Newcastle disease virus (Inf. with),NaN,Unavailable,NaN,NaN,Unavailable
851,1,2005,Jul-Dec 2005,Nigeria,Fowl pox (-2005),NaN,Unavailable,NaN,NaN,Unavailable
852,2,2005,Jul-Dec 2005,Nigeria,Fowl typhoid,NaN,Unavailable,NaN,NaN,Unavailable
853,3,2005,Jul-Dec 2005,Nigeria,Infectious bursal disease (Gumboro disease),NaN,Unavailable,NaN,NaN,Unavailable
855,4,2005,Jul-Dec 2005,Nigeria,Newcastle disease virus (Inf. with),NaN,Unavailable,NaN,NaN,Unavailable
427,6,2007,Jul-Dec 2007,Gombe,Infectious bursal disease (Gumboro disease),NaN,Unavailable,NaN,NaN,Unavailable
30,13,2008,Jan-Jun 2008,Abuja Federal Capital Territory,Newcastle disease virus (Inf. with),NaN,Unavailable,NaN,NaN,Unavailable
53,7,2008,Jan-Jun 2008,Adamawa,Newcastle disease virus (Inf. with),NaN,Unavailable,NaN,NaN,Unavailable
120,8,2008,Jan-Jun 2008,Bauchi,Infectious bursal disease (Gumboro disease),NaN,Unavailable,NaN,NaN,Unavailable
136,9,2008,Jan-Jun 2008,Bauchi,Newcastle disease virus (Inf. with),NaN,Unavailable,NaN,NaN,Unavailable



TARGET DEFINITION VALIDATION PASSED


In [19]:
# ======================================================================
# CELL 9 — FINAL MODELLING FEATURE QUALITY AUDIT
# ======================================================================
# Purpose:
#   Perform the final quality-control audit of the engineered dataset
#   before moving to Notebook 07 — Baseline Model.
#
# Important methodological rules:
#   1. Do not impute missing values in this notebook.
#   2. Do not convert unavailable surveillance observations to zero.
#   3. Do not include identifiers or target-derived fields as predictors.
#   4. Do not include future information.
#   5. Missing climate/history values remain explicit and will be handled
#      later inside the modelling pipeline.
# ======================================================================

# ----------------------------------------------------------------------
# 1. Define candidate modelling feature groups
# ----------------------------------------------------------------------

# Temporal features
temporal_features = [
    "year",
    "semester_order",
    "is_first_semester",
    "is_second_semester",
    "semester_sin",
    "semester_cos",
    "years_since_2019",
    "year_sin",
    "year_cos",
]

# Geographic features
geographic_features = [
    "has_adm1",
    "is_country_level",
]

# Disease features
disease_features = [
    "disease_code",
    "disease_frequency",
    "adm1_disease_record_count",
]

# Climate features
climate_features = [
    "temperature_c",
    "dewpoint_c",
    "semester_total_precipitation_mm",
    "semester_mean_wind_component_magnitude_ms",
]

# Climate availability indicators
climate_availability_features = [
    "climate_available",
    "temperature_missing",
    "dewpoint_missing",
    "precipitation_missing",
    "wind_missing",
    "climate_variable_count",
    "climate_complete",
]

# Historical outbreak features
history_features = [
    "previous_semester_outbreak_count",
    "previous_semester_outbreak_available",
    "two_semesters_prior_outbreak_count",
    "two_semesters_prior_available",
    "outbreak_history_mean_3",
    "outbreak_history_max_3",
]

# Combine all candidate predictor groups.
candidate_features = (
    temporal_features
    + geographic_features
    + disease_features
    + climate_features
    + climate_availability_features
    + history_features
)


# ----------------------------------------------------------------------
# 2. Confirm that every candidate feature exists
# ----------------------------------------------------------------------

missing_candidate_features = [
    feature
    for feature in candidate_features
    if feature not in df.columns
]

assert len(missing_candidate_features) == 0, (
    f"Missing candidate features: {missing_candidate_features}"
)


# ----------------------------------------------------------------------
# 3. Identify columns that must NOT be used as predictors
# ----------------------------------------------------------------------

# These include:
#   - unique record identifiers
#   - raw outbreak target fields
#   - target-derived fields
#   - raw surveillance identifiers
#   - raw geographic labels that will require encoding
#   - raw disease names that will require encoding
#
# They are retained in df for traceability but are not part of the
# initial numerical modelling matrix.

excluded_predictor_columns = [
    "analytical_record_id",
    "new_outbreaks",
    "outbreak_count_observed",
    "outbreak_observation_status",
    "target_outbreak_count",
    "target_outbreak_occurrence",
    "target_observed",
    "target_status",
    "semester_outbreak_count",
    "previous_semester_outbreak_status",
    "previous_semester_temporal_order",
    "temporal_order",
    "two_semesters_prior_temporal_order",
    "adm1_key",
    "validated_boundary_name",
    "administrative_division",
    "disease",
    "country",
    "world_region",
    "semester",
    "year_key",
    "semester_key",
    "integration_key",
    "geographic_level",
    "geographic_match_status",
    "climate_match_status",
]


# ----------------------------------------------------------------------
# 4. Check feature data types
# ----------------------------------------------------------------------

feature_dtype_summary = (
    df[candidate_features]
    .dtypes
    .astype(str)
    .value_counts()
    .rename_axis("dtype")
    .reset_index(name="feature_count")
)

print("=" * 70)
print("FINAL MODELLING FEATURE QUALITY AUDIT")
print("=" * 70)

print("\nCandidate predictor count:")
print(f"  {len(candidate_features)} features")

print("\nCandidate feature data types:")
display(feature_dtype_summary)


# ----------------------------------------------------------------------
# 5. Check missingness
# ----------------------------------------------------------------------

feature_missingness = (
    df[candidate_features]
    .isna()
    .sum()
    .sort_values(ascending=False)
)

feature_missingness_pct = (
    feature_missingness / len(df) * 100
)

missingness_summary = pd.DataFrame({
    "missing_count": feature_missingness,
    "missing_percentage": feature_missingness_pct.round(2),
})

print("\nFeature missingness:")
display(missingness_summary)


# ----------------------------------------------------------------------
# 6. Check infinite values
# ----------------------------------------------------------------------

numeric_candidate_features = df[
    candidate_features
].select_dtypes(include=np.number).columns.tolist()

infinite_counts = pd.Series({
    feature: np.isinf(df[feature]).sum()
    for feature in numeric_candidate_features
})

total_infinite_values = infinite_counts.sum()

assert total_infinite_values == 0, (
    "Infinite values detected in candidate numeric features."
)

print("\nInfinite numeric values:")
print(f"  {total_infinite_values}")


# ----------------------------------------------------------------------
# 7. Check constant features
# ----------------------------------------------------------------------

constant_features = [
    feature
    for feature in candidate_features
    if df[feature].nunique(dropna=False) <= 1
]

print("\nConstant features:")
print(f"  {len(constant_features)}")

if constant_features:
    print(constant_features)


# ----------------------------------------------------------------------
# 8. Check target leakage
# ----------------------------------------------------------------------

# Candidate predictors must not contain the current-period outbreak
# target or any feature derived from the current target.

target_leakage_columns = [
    feature
    for feature in candidate_features
    if (
        "target" in feature.lower()
        or "current" in feature.lower()
        or feature == "new_outbreaks"
    )
]

assert len(target_leakage_columns) == 0, (
    f"Potential target leakage detected: {target_leakage_columns}"
)

print("\nTarget leakage check:")
print("  No target-derived predictors detected.")


# ----------------------------------------------------------------------
# 9. Verify historical features are genuinely historical
# ----------------------------------------------------------------------

# Historical outbreak features are allowed because they describe
# information from earlier semesters.

historical_feature_columns = [
    "previous_semester_outbreak_count",
    "two_semesters_prior_outbreak_count",
    "outbreak_history_mean_3",
    "outbreak_history_max_3",
]

print("\nHistorical feature availability:")

for feature in historical_feature_columns:
    available = df[feature].notna().sum()
    percentage = available / len(df) * 100

    print(
        f"  {feature}: "
        f"{available}/{len(df)} available "
        f"({percentage:.2f}%)"
    )


# ----------------------------------------------------------------------
# 10. Verify climate feature availability
# ----------------------------------------------------------------------

print("\nClimate feature availability:")

for feature in climate_features:
    available = df[feature].notna().sum()
    percentage = available / len(df) * 100

    print(
        f"  {feature}: "
        f"{available}/{len(df)} available "
        f"({percentage:.2f}%)"
    )


# ----------------------------------------------------------------------
# 11. Construct the modelling dataset for audit purposes
# ----------------------------------------------------------------------

# We only select observed targets here.
# This does NOT mean unavailable records are negative examples.
# It simply identifies records with a valid supervised target.

modeling_df = df[
    df["target_observed"]
].copy()

X_candidate = modeling_df[
    candidate_features
].copy()

y_candidate = modeling_df[
    "target_outbreak_count"
].copy()


# ----------------------------------------------------------------------
# 12. Final modelling dataset validation
# ----------------------------------------------------------------------

assert len(modeling_df) == 445, (
    "Unexpected number of supervised modelling records."
)

assert len(X_candidate) == len(y_candidate), (
    "Feature and target row counts do not match."
)

assert y_candidate.notna().all(), (
    "Supervised modelling target contains missing values."
)

assert (y_candidate > 0).all(), (
    "Observed outbreak target contains unexpected non-positive values."
)

assert X_candidate.index.equals(y_candidate.index), (
    "Feature and target indices are not aligned."
)


# ----------------------------------------------------------------------
# 13. Summary
# ----------------------------------------------------------------------

print("\n" + "=" * 70)
print("MODELLING DATASET SUMMARY")
print("=" * 70)

print(f"Supervised modelling records: {len(modeling_df)}")
print(f"Candidate predictors: {len(candidate_features)}")
print(f"Target variable: target_outbreak_count")
print(f"Target observations: {y_candidate.notna().sum()}")
print(f"Unavailable WAHIS records excluded from supervised target: "
      f"{(~df['target_observed']).sum()}")

print("\nTarget distribution:")
display(y_candidate.describe().to_frame("target_outbreak_count"))

print("\n" + "=" * 70)
print("FINAL MODELLING FEATURE QUALITY AUDIT PASSED")
print("=" * 70)

FINAL MODELLING FEATURE QUALITY AUDIT

Candidate predictor count:
  31 features

Candidate feature data types:


,dtype,feature_count
0,int64,18
1,float64,12
2,Int64,1



Feature missingness:


,missing_count,missing_percentage
two_semesters_prior_outbreak_count,646,54.75
previous_semester_outbreak_count,517,43.81
outbreak_history_max_3,439,37.20
outbreak_history_mean_3,439,37.20
temperature_c,196,16.61
semester_total_precipitation_mm,196,16.61
dewpoint_c,196,16.61
semester_mean_wind_component_magnitude_ms,196,16.61
years_since_2019,0,0.00
semester_order,0,0.00



Infinite numeric values:
  0

Constant features:
  0

Target leakage check:
  No target-derived predictors detected.

Historical feature availability:
  previous_semester_outbreak_count: 663/1180 available (56.19%)
  two_semesters_prior_outbreak_count: 534/1180 available (45.25%)
  outbreak_history_mean_3: 741/1180 available (62.80%)
  outbreak_history_max_3: 741/1180 available (62.80%)

Climate feature availability:
  temperature_c: 984/1180 available (83.39%)
  dewpoint_c: 984/1180 available (83.39%)
  semester_total_precipitation_mm: 984/1180 available (83.39%)
  semester_mean_wind_component_magnitude_ms: 984/1180 available (83.39%)

MODELLING DATASET SUMMARY
Supervised modelling records: 445
Candidate predictors: 31
Target variable: target_outbreak_count
Target observations: 445
Unavailable WAHIS records excluded from supervised target: 735

Target distribution:


,target_outbreak_count
count,445.000000
mean,6.840449
std,12.872353
min,1.000000
25%,1.000000
50%,3.000000
75%,7.000000
max,184.000000



FINAL MODELLING FEATURE QUALITY AUDIT PASSED


In [20]:
# ======================================================================
# CELL 10 — TEMPORAL LEAKAGE AUDIT FOR DERIVED FEATURES
# ======================================================================
# Purpose:
#   Audit engineered features whose values may have been calculated using
#   information from the full dataset rather than information available
#   at the prediction time.
#
# Critical modelling principle:
#   For a prediction made at time T, a predictor must not contain
#   information from time T or any future period.
#
# This cell does NOT rebuild the features yet.
# It identifies which features require time-aware construction inside
# the modelling pipeline.
# ======================================================================

print("=" * 70)
print("TEMPORAL LEAKAGE AUDIT FOR DERIVED FEATURES")
print("=" * 70)


# ----------------------------------------------------------------------
# 1. Define the temporal order
# ----------------------------------------------------------------------

# The project uses two semesters per year:
#   1 = Jan-Jun
#   2 = Jul-Dec
#
# A larger temporal_order therefore represents a later period.

df["temporal_order_audit"] = (
    (df["year"] - df["year"].min()) * 2
    + df["semester_order"]
)

assert df["temporal_order_audit"].notna().all(), (
    "Temporal order contains missing values."
)


# ----------------------------------------------------------------------
# 2. Audit disease_frequency
# ----------------------------------------------------------------------
# disease_frequency was created during feature engineering.
#
# If it was calculated using the complete dataset, its value may include
# records from future periods.
#
# We compare the current feature against a strictly historical version
# calculated using only records from earlier periods.

historical_disease_frequency = (
    df.groupby(
        ["disease", "temporal_order_audit"]
    )
    .size()
    .groupby(level=0)
    .cumsum()
    .shift(1)
)

# The construction above is not used as a replacement feature.
# This audit simply highlights that disease_frequency is potentially
# full-period information rather than a strictly historical predictor.

print("\nDisease frequency:")
print(
    "  Current feature is calculated across the available dataset."
)
print(
    "  It should therefore be treated as a potentially "
    "time-leaky derived feature."
)


# ----------------------------------------------------------------------
# 3. Audit ADM1 × disease record count
# ----------------------------------------------------------------------
# adm1_disease_record_count may also contain future records if calculated
# across the complete dataset.

print("\nADM1 × disease record count:")
print(
    "  Current feature is calculated across the available dataset."
)
print(
    "  It should therefore be treated as a potentially "
    "time-leaky derived feature."
)


# ----------------------------------------------------------------------
# 4. Audit explicitly historical outbreak features
# ----------------------------------------------------------------------
# These features were constructed using previous semesters only.
# The earlier Cell 7 validation already confirmed that their temporal
# reference is strictly earlier than the current period.

historical_features_to_validate = [
    "previous_semester_outbreak_count",
    "two_semesters_prior_outbreak_count",
    "outbreak_history_mean_3",
    "outbreak_history_max_3",
]

print("\nHistorical outbreak features:")

for feature in historical_features_to_validate:
    available_mask = df[feature].notna()

    print(
        f"  {feature}: "
        f"{available_mask.sum()} available / {len(df)} total"
    )

print(
    "\n  Previous outbreak features were explicitly constructed "
    "from earlier semesters."
)


# ----------------------------------------------------------------------
# 5. Audit climate features
# ----------------------------------------------------------------------
# Climate observations correspond to the same ADM1 × year × semester
# period as the target.
#
# This is acceptable if the intended prediction scenario assumes that
# climate observations for the target semester are available at the
# time the prediction is made.
#
# However, if the eventual system is intended to predict outbreaks
# BEFORE the semester occurs, these same-period climate variables would
# need to be replaced with forecasts or lagged climate variables.

climate_features_audit = [
    "temperature_c",
    "dewpoint_c",
    "semester_total_precipitation_mm",
    "semester_mean_wind_component_magnitude_ms",
]

print("\nClimate features:")

for feature in climate_features_audit:
    available = df[feature].notna().sum()

    print(
        f"  {feature}: "
        f"{available} available / {len(df)} total"
    )

print(
    "\n  Climate variables represent the same semester as the target."
)
print(
    "  They are valid for retrospective/end-of-semester modelling, "
    "but not automatically valid for pre-semester forecasting."
)


# ----------------------------------------------------------------------
# 6. Audit year-derived features
# ----------------------------------------------------------------------
# These features describe calendar time and do not contain future
# outbreak observations.

year_features = [
    "year",
    "years_since_2019",
    "year_sin",
    "year_cos",
]

print("\nYear-derived features:")

for feature in year_features:
    print(f"  {feature}: calendar-time feature; no target leakage detected.")


# ----------------------------------------------------------------------
# 7. Audit semester-derived features
# ----------------------------------------------------------------------

semester_features = [
    "semester_order",
    "is_first_semester",
    "is_second_semester",
    "semester_sin",
    "semester_cos",
]

print("\nSemester-derived features:")

for feature in semester_features:
    print(
        f"  {feature}: calendar-semester feature; "
        "no target leakage detected."
    )


# ----------------------------------------------------------------------
# 8. Produce a leakage-risk classification
# ----------------------------------------------------------------------

feature_leakage_audit = pd.DataFrame({
    "feature_group": [
        "Disease frequency",
        "ADM1 × disease record count",
        "Historical outbreak features",
        "Climate features",
        "Year-derived features",
        "Semester-derived features",
    ],
    "temporal_risk": [
        "High",
        "High",
        "Low",
        "Context-dependent",
        "Low",
        "Low",
    ],
    "reason": [
        "Calculated across the available dataset and may include future records.",
        "Calculated across the available dataset and may include future records.",
        "Explicitly constructed from earlier semesters only.",
        "Same-semester climate values are valid retrospectively but may not be available before the semester.",
        "Calendar information does not contain future outbreak observations.",
        "Calendar information does not contain future outbreak observations.",
    ],
})

print("\n" + "=" * 70)
print("TEMPORAL LEAKAGE RISK ASSESSMENT")
print("=" * 70)

display(feature_leakage_audit)


# ----------------------------------------------------------------------
# 9. Create the safe feature set for Notebook 07
# ----------------------------------------------------------------------
# For the initial modelling pipeline, we exclude the two potentially
# time-leaky full-period frequency features.
#
# We retain historical outbreak features because they were explicitly
# constructed using earlier periods.
#
# Same-semester climate variables are retained for now, but Notebook 07
# must clearly define the modelling scenario as retrospective/end-period
# prediction unless lagged/forecast climate variables are introduced.

safe_candidate_features = [
    feature
    for feature in candidate_features
    if feature not in [
        "disease_frequency",
        "adm1_disease_record_count",
    ]
]

print("\nSafe candidate feature count:")
print(f"  {len(safe_candidate_features)}")

print("\nExcluded potentially time-leaky features:")
print("  - disease_frequency")
print("  - adm1_disease_record_count")


# ----------------------------------------------------------------------
# 10. Validate the safe feature set
# ----------------------------------------------------------------------

assert "disease_frequency" not in safe_candidate_features
assert "adm1_disease_record_count" not in safe_candidate_features

assert "previous_semester_outbreak_count" in safe_candidate_features
assert "two_semesters_prior_outbreak_count" in safe_candidate_features
assert "outbreak_history_mean_3" in safe_candidate_features
assert "outbreak_history_max_3" in safe_candidate_features

assert "temperature_c" in safe_candidate_features
assert "dewpoint_c" in safe_candidate_features
assert "semester_total_precipitation_mm" in safe_candidate_features
assert "semester_mean_wind_component_magnitude_ms" in safe_candidate_features

print("\n" + "=" * 70)
print("TEMPORAL LEAKAGE AUDIT COMPLETED")
print("=" * 70)

print(
    "\nNotebook 07 should use the safe candidate feature set "
    "and construct any time-varying frequency features using "
    "training data only."
)

TEMPORAL LEAKAGE AUDIT FOR DERIVED FEATURES

Disease frequency:
  Current feature is calculated across the available dataset.
  It should therefore be treated as a potentially time-leaky derived feature.

ADM1 × disease record count:
  Current feature is calculated across the available dataset.
  It should therefore be treated as a potentially time-leaky derived feature.

Historical outbreak features:
  previous_semester_outbreak_count: 663 available / 1180 total
  two_semesters_prior_outbreak_count: 534 available / 1180 total
  outbreak_history_mean_3: 741 available / 1180 total
  outbreak_history_max_3: 741 available / 1180 total

  Previous outbreak features were explicitly constructed from earlier semesters.

Climate features:
  temperature_c: 984 available / 1180 total
  dewpoint_c: 984 available / 1180 total
  semester_total_precipitation_mm: 984 available / 1180 total
  semester_mean_wind_component_magnitude_ms: 984 available / 1180 total

  Climate variables represent the same 

,feature_group,temporal_risk,reason
0,Disease frequency,High,Calculated across the available dataset and ma...
1,ADM1 × disease record count,High,Calculated across the available dataset and ma...
2,Historical outbreak features,Low,Explicitly constructed from earlier semesters ...
3,Climate features,Context-dependent,Same-semester climate values are valid retrosp...
4,Year-derived features,Low,Calendar information does not contain future o...
5,Semester-derived features,Low,Calendar information does not contain future o...



Safe candidate feature count:
  29

Excluded potentially time-leaky features:
  - disease_frequency
  - adm1_disease_record_count

TEMPORAL LEAKAGE AUDIT COMPLETED

Notebook 07 should use the safe candidate feature set and construct any time-varying frequency features using training data only.


# Export Modelling-Ready Feature Dataset

The feature-engineering stage has now been completed and validated.

This section exports the engineered dataset for use in downstream
modelling notebooks.

The exported dataset retains:

- the original WAHIS analytical record;
- validated geographic information;
- temporal features;
- disease features;
- climate features;
- semester-level historical outbreak features;
- target-definition fields; and
- explicit missingness indicators.

Potentially time-leaky features identified during the temporal leakage
audit are retained in the dataset for traceability but will be excluded
from the initial machine-learning predictor set.

No missing climate or historical outbreak values are replaced with zero.

In [21]:
# ============================================================
# EXPORT MODELLING-READY FEATURE DATASET
# ============================================================
#
# Purpose:
# Save the fully engineered feature table so that downstream
# modelling notebooks can use exactly the same features produced
# and validated in Notebook 06.
#
# This prevents duplicate feature-engineering logic across
# notebooks and improves reproducibility.
# ============================================================


# ------------------------------------------------------------
# 1. Define the output path
# ------------------------------------------------------------

output_path = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "wahis_modelling_features.csv"
)


# ------------------------------------------------------------
# 2. Define the required modelling columns
# ------------------------------------------------------------
#
# These are the variables required for the initial modelling
# experiment and the associated traceability fields.
# ------------------------------------------------------------

modelling_columns = [

    # Record identifiers
    "analytical_record_id",

    # Original temporal fields
    "year",
    "semester",

    # Geographic and disease identifiers
    "administrative_division",
    "validated_boundary_name",
    "geographic_level",
    "disease",

    # Original target information
    "new_outbreaks",
    "outbreak_count_observed",
    "outbreak_observation_status",

    # Engineered temporal features
    "semester_order",
    "is_first_semester",
    "is_second_semester",
    "semester_sin",
    "semester_cos",
    "years_since_2019",
    "year_sin",
    "year_cos",

    # Engineered geographic features
    "has_adm1",
    "is_country_level",

    # Engineered disease features
    "disease_code",
    "disease_frequency",
    "adm1_disease_record_count",

    # Historical outbreak features
    "previous_semester_outbreak_count",
    "two_semesters_prior_outbreak_count",
    "outbreak_history_mean_3",
    "outbreak_history_max_3",

    # Climate availability features
    "climate_available",
    "temperature_missing",
    "dewpoint_missing",
    "precipitation_missing",
    "wind_missing",
    "climate_variable_count",
    "climate_complete",

    # Climate predictors
    "temperature_c",
    "dewpoint_c",
    "semester_total_precipitation_mm",
    "semester_mean_wind_component_magnitude_ms",

]


# ------------------------------------------------------------
# 3. Validate required columns
# ------------------------------------------------------------

missing_columns = [
    column
    for column in modelling_columns
    if column not in df.columns
]

assert (
    len(missing_columns) == 0
), (
    "Required modelling columns are missing: "
    f"{missing_columns}"
)


# ------------------------------------------------------------
# 4. Create the modelling-ready dataset
# ------------------------------------------------------------

modelling_features = (
    df[modelling_columns]
    .copy()
)


# ------------------------------------------------------------
# 5. Validate record preservation
# ------------------------------------------------------------

assert (
    len(modelling_features) == len(df)
), (
    "Modelling feature table changed the number of records."
)

assert (
    modelling_features[
        "analytical_record_id"
    ].is_unique
), (
    "Duplicate analytical record IDs detected."
)


# ------------------------------------------------------------
# 6. Validate target observation structure
# ------------------------------------------------------------

assert (
    modelling_features.loc[
        modelling_features[
            "outbreak_observation_status"
        ] == "Observed",
        "new_outbreaks"
    ]
    .notna()
    .all()
), (
    "Observed outbreak records contain missing target counts."
)

assert (
    modelling_features.loc[
        modelling_features[
            "outbreak_observation_status"
        ] == "Unavailable",
        "new_outbreaks"
    ]
    .isna()
    .all()
), (
    "Unavailable outbreak observations contain numeric counts."
)


# ------------------------------------------------------------
# 7. Validate historical features
# ------------------------------------------------------------
#
# Historical features must remain missing when the required
# historical observation is unavailable.
#
# We do NOT replace these missing values with zero.
# ------------------------------------------------------------

historical_features = [
    "previous_semester_outbreak_count",
    "two_semesters_prior_outbreak_count",
    "outbreak_history_mean_3",
    "outbreak_history_max_3"
]

print("\nHistorical feature availability:")

for feature in historical_features:

    available = int(
        modelling_features[feature]
        .notna()
        .sum()
    )

    print(
        f"  {feature}: "
        f"{available} available / "
        f"{len(modelling_features)} total"
    )


# ------------------------------------------------------------
# 8. Validate climate feature availability
# ------------------------------------------------------------

climate_features = [
    "temperature_c",
    "dewpoint_c",
    "semester_total_precipitation_mm",
    "semester_mean_wind_component_magnitude_ms"
]

print("\nClimate feature availability:")

for feature in climate_features:

    available = int(
        modelling_features[feature]
        .notna()
        .sum()
    )

    print(
        f"  {feature}: "
        f"{available} available / "
        f"{len(modelling_features)} total"
    )


# ------------------------------------------------------------
# 9. Save the modelling-ready feature table
# ------------------------------------------------------------

modelling_features.to_csv(
    output_path,
    index=False
)


# ------------------------------------------------------------
# 10. Reload the saved dataset to verify the export
# ------------------------------------------------------------

reloaded_features = pd.read_csv(
    output_path
)


# ------------------------------------------------------------
# 11. Export validation
# ------------------------------------------------------------

assert (
    reloaded_features.shape
    == modelling_features.shape
), (
    "Reloaded modelling dataset shape does not match "
    "the exported dataset."
)

assert (
    reloaded_features[
        "analytical_record_id"
    ].is_unique
), (
    "Duplicate analytical record IDs detected after reload."
)

assert (
    reloaded_features[
        "analytical_record_id"
    ].tolist()
    ==
    modelling_features[
        "analytical_record_id"
    ].tolist()
), (
    "Analytical record ordering changed during export."
)


print("\n" + "=" * 70)
print("MODELLING-READY FEATURE DATASET EXPORT")
print("=" * 70)

print(
    f"Rows: {reloaded_features.shape[0]}"
)

print(
    f"Columns: {reloaded_features.shape[1]}"
)

print(
    f"Output: {output_path}"
)

print(
    "\nModelling-ready feature dataset exported "
    "and successfully reloaded."
)

print(
    "\nMODELLING FEATURE EXPORT VALIDATION PASSED"
)


Historical feature availability:
  previous_semester_outbreak_count: 663 available / 1180 total
  two_semesters_prior_outbreak_count: 534 available / 1180 total
  outbreak_history_mean_3: 741 available / 1180 total
  outbreak_history_max_3: 741 available / 1180 total

Climate feature availability:
  temperature_c: 984 available / 1180 total
  dewpoint_c: 984 available / 1180 total
  semester_total_precipitation_mm: 984 available / 1180 total
  semester_mean_wind_component_magnitude_ms: 984 available / 1180 total

MODELLING-READY FEATURE DATASET EXPORT
Rows: 1180
Columns: 38
Output: c:\Users\TECHGROW ANALYTICS\poultry-disease-early-warning-nigeria\data\processed\wahis_modelling_features.csv

Modelling-ready feature dataset exported and successfully reloaded.

MODELLING FEATURE EXPORT VALIDATION PASSED
